# 05 · Feature Engineering

Engineer `Time` into cyclical hour features and `Amount` into a log transform, then test honestly whether they lift PR-AUC over the raw features on a held-out validation set.

In [1]:
# --- project-root bootstrap: portable across VS Code / Jupyter / CLI ---
import os
from pathlib import Path
_p = Path.cwd()
while not (_p / 'config' / 'config.yaml').exists() and _p != _p.parent:
    _p = _p.parent
os.chdir(_p)
print('working dir:', Path.cwd())

working dir: /Users/asfalanoi/app_2026/fraud_detection


In [2]:
import numpy as np
import pandas as pd
from fraud.io import read_parquet
from fraud.features import CyclicalHour, AmountTransforms

train = read_parquet('data/processed/train.parquet')
valid = read_parquet('data/processed/valid.parquet')
ch = CyclicalHour().fit_transform(train[['Time']])
at = AmountTransforms().fit_transform(train[['Amount']])
pd.DataFrame({'hour_sin': ch[:, 0], 'hour_cos': ch[:, 1], 'log_amount': at[:, 0]}).head()

,hour_sin,hour_cos,log_amount
0,-0.711260,0.702929,2.118662
1,0.363996,-0.931400,1.383791
2,0.145579,-0.989347,5.171052
3,-0.218143,-0.975917,1.960095
4,-0.922734,-0.385436,4.467057


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import average_precision_score

Vc = [f'V{i}' for i in range(1, 29)]

def fit_score(Xtr, ytr, Xva, yva):
    m = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=1000, class_weight='balanced'))
    m.fit(Xtr, ytr)
    return average_precision_score(yva, m.predict_proba(Xva)[:, 1])

def engineer(df):
    ch = CyclicalHour().fit_transform(df[['Time']])
    at = AmountTransforms().fit_transform(df[['Amount']])
    out = df[Vc].copy()
    out['log_amount'] = at[:, 0]
    out['hour_sin'] = ch[:, 0]
    out['hour_cos'] = ch[:, 1]
    return out

In [4]:
pr_raw = fit_score(train[Vc + ['Amount', 'Time']], train.Class,
                   valid[Vc + ['Amount', 'Time']], valid.Class)
pr_eng = fit_score(engineer(train), train.Class, engineer(valid), valid.Class)
print(f'PR-AUC raw       = {pr_raw:.4f}')
print(f'PR-AUC engineered= {pr_eng:.4f}')
print(f'lift             = {pr_eng - pr_raw:+.4f}')

PR-AUC raw       = 0.6695
PR-AUC engineered= 0.6624
lift             = -0.0071


**Honest read:** on a linear model the engineered features give a *negative* lift (about -0.007 PR-AUC). V1-V28 already capture the signal and logistic regression cannot exploit the cyclical encodings. We carry them into `06_modeling` only to test whether the gradient-boosted model (which can model interactions) benefits, and because hour and log-amount are interpretable for the business narrative. **If they do not earn their keep on the tree model either, we drop them** — engineering that does not improve the decision is removed.